# Forecasting a London household's next 24 hours of electricity use with an LSTM and a GRU

**Patrick Gilliland**, AI Mastery Capstone: Deep Learning Systems

This is a sequential or time-series task using a Recurrent Neural Network (RNN). From one household's previous seven days of half-hourly readings, the notebook forecasts that household's next 24 hours (48 half-hourly readings, in kWh). Two recurrent models make the forecast, and they differ only in the layer type: the baseline model (the LSTM) and the experimental model (the GRU). The data is the Low Carbon London smart meter data (UK Power Networks), 1,000 households from 2013. The report ships as `Deep_Learning_Systems_Analysis_Report.pdf` and, with the same content, as `module_summary.pdf`.

**Setup notes**

| Item | This notebook |
|---|---|
| Rig | NVIDIA GeForce RTX 5090, Linux under WSL2 |
| Python | 3.12.3 |
| PyTorch | 2.14.1 from PyPI |
| CUDA | 13.0 (the runtime PyTorch ships with), cuDNN 9.24.0 |
| Training time on the GPU, last full run | [TO FILL from the Phase 2 full run] |
| Estimated training time on a CPU | [TO FILL from the Phase 2 full run] |

Both times come from the last full run of this notebook, and the code prints them in Task 5's timing table. With no GPU visible, the same code runs on the CPU.

**Setup.** The first code cell sets two environment variables before PyTorch loads, seeds every random number generator, fixes the settings that make a rerun repeat its numbers, caps the GPU's memory when there is a GPU, and defines the training watchdog that Task 3 uses.

In [1]:
import math
import os
import platform
import random
import re
import statistics

# A value set outside the notebook wins, so CUDA_VISIBLE_DEVICES set to an empty string hides
# the GPU for a CPU run. The cuBLAS setting is the one PyTorch's nn.LSTM page gives for
# repeatable recurrent layers on a GPU.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:2")

import numpy as np  # noqa: E402
import torch  # noqa: E402

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Deterministic cuDNN kernels and no TF32, so a rerun gives the same numbers and the arithmetic
# stays float32.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

on_gpu = torch.cuda.is_available()
if on_gpu:
    # Half the card at most, by this rig's GPU rules.
    torch.cuda.set_per_process_memory_fraction(0.5)


def watchdog(epoch_seconds, loss, elapsed_seconds, on_gpu, factor=3.0, limit_seconds=7200.0):
    """Return why a training run should stop, or None to carry on.

    epoch_seconds lists each finished epoch's seconds, the latest last. A NaN or infinite loss
    stops a run on any device. On the GPU, an epoch slower than factor times the median of the
    epochs before it, or a run past limit_seconds, also stops it. On a CPU the run goes to the
    end, because a grader's CPU run is meant to finish.
    """
    if not math.isfinite(loss):
        return f"loss is {loss}"
    if not on_gpu:
        return None
    if elapsed_seconds > limit_seconds:
        return f"the run passed {limit_seconds / 3600:g} hours"
    if len(epoch_seconds) > 1:
        line = factor * statistics.median(epoch_seconds[:-1])
        if epoch_seconds[-1] > line:
            return f"the epoch took {epoch_seconds[-1]:.1f} s, over {line:.1f} s"
    return None


print(f"Python {platform.python_version()}, PyTorch {torch.__version__}, NumPy {np.__version__}")
print(f"Platform: {platform.platform()}")
print(f"Seed: {SEED}")
print(f"CUDA_VISIBLE_DEVICES={os.environ['CUDA_VISIBLE_DEVICES']!r}, "
      f"CUBLAS_WORKSPACE_CONFIG={os.environ['CUBLAS_WORKSPACE_CONFIG']!r}")
if on_gpu:
    cudnn = torch.backends.cudnn.version()
    print(f"Device: cuda, {torch.cuda.get_device_name(0)}, CUDA {torch.version.cuda}, "
          f"cuDNN {cudnn // 10000}.{cudnn % 10000 // 100}.{cudnn % 100}")
    print("GPU memory cap: half the card")
else:
    print("Device: cpu, no GPU visible, so the watchdog lets a run go to the end")

Python 3.12.3, PyTorch 2.14.1+cu130, NumPy 2.5.4
Platform: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
Seed: 42
CUDA_VISIBLE_DEVICES='0', CUBLAS_WORKSPACE_CONFIG=':4096:2'
Device: cuda, NVIDIA GeForce RTX 5090, CUDA 13.0, cuDNN 9.24.0
GPU memory cap: half the card


**Environment check.** The brief's own validation code, as given, reading this project's data file. In one cell only the last line's value is shown, so `device` on its own line prints nothing here; the setup cell above prints the device.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
device
df = pd.read_csv("lcl_2013_1000_households.csv.gz")
df.head()

,LCLid,tariff_group,date,price_signal,r00,r01,r02,r03,r04,r05,...,r38,r39,r40,r41,r42,r43,r44,r45,r46,r47
0,MAC000006,Std,2013-01-01,NaN,0.0,0.0,0.0,0.0,0.0,0.0,...,0.044,0.044,0.043,0.044,0.030,0.0,0.0,0.0,0.0,0.0
1,MAC000006,Std,2013-01-02,NaN,0.0,0.0,0.0,0.0,0.0,0.0,...,0.044,0.043,0.043,0.044,0.029,0.0,0.0,0.0,0.0,0.0
2,MAC000006,Std,2013-01-03,NaN,0.0,0.0,0.0,0.0,0.0,0.0,...,0.043,0.043,0.043,0.043,0.024,0.0,0.0,0.0,0.0,0.0
3,MAC000006,Std,2013-01-04,NaN,0.0,0.0,0.0,0.0,0.0,0.0,...,0.043,0.044,0.043,0.043,0.030,0.0,0.0,0.0,0.0,0.0
4,MAC000006,Std,2013-01-05,NaN,0.0,0.0,0.0,0.0,0.0,0.0,...,0.043,0.043,0.043,0.043,0.030,0.0,0.0,0.0,0.0,0.0


## Task 1: Define the Problem and Choose Your Dataset

**Goal.** Show where the data came from and what the pipeline did to it, from the manifest the pipeline wrote. Then state the task, the question, and why the dataset meets each of the brief's rules.

In [3]:
def manifest_tables(text):
    """Return the tables in MANIFEST.md text, keyed by the heading of the section they sit in.

    Each value is a list of that section's tables, in order, as DataFrames of strings. A section
    with no table is left out.
    """
    tables, heading, rows = {}, None, []
    for line in text.splitlines() + [""]:
        if line.startswith("|"):
            rows.append([cell.strip() for cell in line.strip().strip("|").split("|")])
            continue
        if rows and heading is not None:
            separator = all(re.fullmatch(r":?-+:?", cell) for cell in rows[1])
            body = rows[2:] if separator else rows[1:]
            tables.setdefault(heading, []).append(pd.DataFrame(body, columns=rows[0]))
        rows = []
        if line.startswith("## "):
            heading = line[3:].strip()
    return tables

In [4]:
with open("data_pipeline/MANIFEST.md", encoding="utf-8") as fh:
    manifest_text = fh.read()
manifest = manifest_tables(manifest_text)

print(next(line for line in manifest_text.splitlines() if line.startswith("Written by")))
for section in ["Source", "Files", "The publisher's figures beside the measured ones",
                "Pipeline steps", "Price signals"]:
    print(f"\n{section}")
    for table in manifest[section]:
        print(table.to_string(index=False))

steps = manifest["Pipeline steps"][0].set_index("Step")["Count"]
reading_columns = [column for column in df.columns if re.fullmatch(r"r\d\d", column)]
assert len(df) == int(steps["Output rows"].replace(",", "")), "the file and the manifest differ"
print(f"\nThe working file the validation cell loaded: {len(df):,} rows, as the manifest's output "
      f"rows; {len(reading_columns)} half-hourly readings a row ({len(reading_columns) / 2:g} "
      f"hours), {len(df) * len(reading_columns):,} readings in all")

Written by `data_pipeline/build_lcl_subset.py` on 2026-10-11, with Python 3.12.3, pandas 3.0.6 and NumPy 2.5.4. Year 2013, 1,000 households, seed 42, pool `strict`.

Source
         Item                                                                                           Value
     Data set                                         SmartMeter Energy Consumption Data in London Households
    Publisher                          UK Power Networks (Low Carbon London project), on the London Datastore
         Page              https://data.london.gov.uk/dataset/smartmeter-energy-use-data-in-london-households
      Licence          Creative Commons Attribution (CC BY 4.0), https://creativecommons.org/licenses/by/4.0/
Readings file https://data.london.gov.uk/download/vqm0d/3527bf39-d93e-4071-8451-df2ade1ea4f2/LCL-FullData.zip
  Tariff file     https://data.london.gov.uk/download/vqm0d/14855047-44c2-4856-8a48-e5649200e6ce/Tariffs.xlsx
   Downloaded                                            

**The task.** Of the brief's three task types, this is the second: "Sequential or time-series tasks using a Recurrent Neural Network (RNN)". The task is forecasting one household's next 24 hours (48 half-hourly readings), each a half-hourly reading in kWh.

**The question.** From one household's previous seven days of half-hourly readings, how well can a Recurrent Neural Network forecast its next 24 hours (48 half-hourly readings), and does the experimental model (the GRU) forecast better or worse than the baseline model (the LSTM) when the layer type is the only change?

**The dataset.**

| Item | Value |
|---|---|
| Name | SmartMeter Energy Consumption Data in London Households: the Low Carbon London smart meter data (UK Power Networks) |
| Publisher | UK Power Networks (Low Carbon London project), on the London Datastore |
| Page | https://data.london.gov.uk/dataset/smartmeter-energy-use-data-in-london-households |
| Licence | Creative Commons Attribution (CC BY 4.0) |
| File | `lcl_2013_1000_households.csv.gz`, beside this notebook, built from the download by `data_pipeline/build_lcl_subset.py` |
| Size | 26,957,559 bytes (25.71 MiB) as gzip; the CSV inside is 113,142,976 bytes |
| SHA-256 | `9fd762e499e60c7bc17db319282707b34864f8b9a3478dd872d9c2037a399c49` |
| Downloaded | 11 October 2026 |

**Why it meets each of the brief's rules.**

- "Be appropriate for deep learning in your chosen domain": each household's readings are an ordered series at a fixed half-hourly step, the kind of sequence a Recurrent Neural Network reads one step at a time.
- "Contain enough samples to support training and evaluation": 1,000 households over a whole year give 365,000 household-days and 17,520,000 readings. Task 2 counts the household-days in the training, validation and test months.
- "Be publicly available and appropriate for academic use": the London Datastore publishes it under the Creative Commons Attribution licence, which allows reuse with credit.
- "Not be synthetic or AI-generated": the publisher describes the readings as "Energy consumption readings for a sample of 5,567 London Households that took part in the UK Power Networks led Low Carbon London project", so they are the trial's own meter readings.
- "Not be reused from any previous capstone project": the earlier capstone projects used Birmingham City Council purchase card data, the GOV.UK payment practices reports and the Department for Transport's spending data. None of them comes from UK Power Networks.

**How the subset was obtained and prepared.**

1. `LCL-FullData.zip` and `Tariffs.xlsx` were downloaded from the page above on 11 October 2026. The zip is compressed with Deflate64, which Python's `zipfile` cannot open, so `CC_LCL-FullData.csv` is extracted first with a tool that handles it, such as Info-ZIP `unzip` or 7-Zip.
2. `data_pipeline/build_lcl_subset.py`, which uses only the standard library and pandas, then dropped the 5,560 "Null" readings (the same 5,560 rows are stamped off the half-hour grid), kept one of each exact duplicate (115,453 dropped), and kept the 93,024,217 readings stamped in 2013.
3. It found the 1,132 households with every 2013 reading (881 in the standard tariff group, 251 in the time-of-use group) and drew 1,000 of them with seed 42: 781 in the standard tariff group and 219 in the time-of-use group.
4. It wrote one row per household-day: `LCLid` (the household), `tariff_group` (the data's `Std` or `ToU`), `date`, `price_signal` (none, high, low or both on a time-of-use household's day, from `Tariffs.xlsx`, and empty for the standard tariff group), then `r00` to `r47`, the day's 48 half-hourly readings in kWh as published.
5. The time stamps are kept as published, and they behave as GMT all year (Task 2 shows the summer shift this causes). The tariff file appears to follow the UK local clock, so the pipeline moves its summer stamps one hour back to match the readings; the manifest's last table lists the four half-hours that the move leaves with two tariff rows or none.

## Task 2: Load and Inspect Your Dataset

## Task 3: Implement a Baseline Deep Learning Model

## Task 4: Conduct an Experimental Comparison

## Task 5: Evaluate and Compare Model Performance

## Summary